# Phase B — 가설 테이블 + 검증 템플릿

**목적**: ai-feat-294 에서 만든 raw eventRows 시각화 도구를 활용하여, 사람/매크로 패턴을 가설 단위로 발굴·검증.

**입력**:
- `eda_gyeom.ipynb` 셀 60 — Phase B 입력 정리 (SMD 순위 + 분포 형태)
- `outputs/294_raw_trajectory/*.png` — 그룹별 trajectory grid
- `visualizer.ipynb` — 그리드 + 단일 trial deep dive helper

**산출**: 본 노트북은 가설 인덱스. 가설 검증은 셀 2 의 템플릿을 복사해서 가설별로 추가.

## 가설 테이블 (B'' — PNG 시각 검증 시드)

초기 4개 (인수 시점) + eda_gyeom 정리에서 도출한 4개 = 총 8개. 발굴 진행 중 자유 추가.

**검증 상태 표기**: `unverified` / `partial` (raw 관찰과 부분 일치) / `verified` / `rejected` (raw 관찰과 불일치) / `trivial` (수집/매크로 알고리즘 산물 의심)

| ID | 가설 | 입력 출처 | 예상 매칭 metric (eda_gyeom n=652 기준) | 검증 상태 |
|---|---|---|---|---|
| H1 | **macro 좌표 영역 사용 범위가 좁다** | trajectory PNG (lv2_human vs lv2_macro grid) | `mouse_total_travel_distance_px` 사람 8643 / 매크로 2694 (3×) | unverified |
| H2 | **macro click 간격이 짧고 일정하다** | speed/dt grid PNG | `inter_click_interval_ms` 사람 7826 / 매크로 1190 (매크로 SMD 0.36 small 이지만 평균치 6.5×) | unverified |
| H3 | **사람 path 가 더 각진/떨림 있다** (직관과 반대 — 매크로 = 직선) | trajectory + speed PNG | `mouse_path_curvature_mean` 0.178 / 0.107, `mouse_jerk_mean` 0.033 / 7e-5 (~470×) | unverified |
| H4 | **Balabit 분산 ≫ lv2_human** (게이트 가설) | balabit grid vs lv2_human grid PNG | trajectory 다양성 시각 비교 — 사람 도메인 일반화 가능 여부 | unverified |
| H5 | **macro 평균 속도가 사람보다 빠르다** (직관과 반대 — H1 의 모순?) | speed grid PNG | `mouse_avg_speed_px_per_ms` 사람 0.17 / 매크로 0.23 (매크로 우세) | unverified |
| H6 | **macro path 가 사람보다 더 직선** | trajectory PNG | `mouse_path_straightness_score` 사람 0.066 / 매크로 0.091 (매크로 우세 — H1/H3 와 정합) | unverified |
| H7 | **macro 의 max speed 가 매우 작다** (Bezier 속도 cap 의심) | speed grid PNG (peak 영역) | `mouse_max_speed_px_per_ms` 사람 7.97 / 매크로 1.12 (~7×) | unverified |
| H8 | **pre-click 영역에서 사람의 hover 시간이 더 김** | pre_click grid PNG | `mouse_hover_dwell_time_ms` 사람 845 / 매크로 484 (1.7×) | unverified |

**상호 관계 메모**
- H1 + H7 = macro 가 좁은 영역에서 천천히 움직임 (단, H5 와 모순 — total_distance 짧아도 평균 속도 빠른 이유?)
- H3 + H6 = path 형태가 정반대 (사람 곡선/떨림, macro 직선) — 일관됨
- H4 = lv2 단일 사용자 vs Balabit 10명, 분산 차이 보이면 사람 도메인 일반화 근거

## 검증 셀 템플릿 (B''' — 가설별 복사)

각 가설 검증 시 아래 3 셀을 복사해서 사용. `metric_name` / `human_id` / `macro_id` / 가설 ID 만 채워넣으면 됩니다.

**구성**
- ① metric 분포 비교 (사람 vs macro) — boxplot
- ② 사람/macro trial 1개씩 raw plot (4 plot 2 row) — plots.py 활용
- ③ "metric 값이 raw 관찰과 일치하는가" 1줄 코멘트 마크다운

In [ ]:
# 템플릿 — 모든 검증 셀의 공통 import (1회만 실행)
%matplotlib inline
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from trial_loader import (
    DATA_DIR, GROUPS,
    list_trials_by_group, load_trial,
)
from plots import (
    plot_trajectory, plot_speed_over_time,
    plot_dt_distribution, plot_pre_click_paths,
)

In [ ]:
# === H? 검증 — Step ① metric 분포 비교 ===
# TODO: metric_name 채우기
metric_name = "TODO_metric_name"  # 예: "mouse_total_travel_distance_px"

rows = []
for path in sorted(DATA_DIR.glob("trial_*.json")):
    d = json.loads(path.read_text(encoding="utf-8"))
    label = d.get("label")
    if label not in ("human", "macro"):
        continue
    val = (d.get("metrics") or {}).get(metric_name)
    if val is not None:
        rows.append({"label": label, metric_name: val})
metric_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=metric_df, x="label", y=metric_name, hue="label",
            order=["human", "macro"], legend=False, dodge=False, ax=ax)
sns.stripplot(data=metric_df, x="label", y=metric_name,
              order=["human", "macro"], color="black", alpha=0.25, size=2, jitter=0.2, ax=ax)
ax.set_title(f"{metric_name}  (n_human={metric_df.label.eq('human').sum()}, n_macro={metric_df.label.eq('macro').sum()})")
plt.tight_layout()
plt.show()

print(metric_df.groupby("label")[metric_name].describe()[["count", "mean", "std", "min", "50%", "max"]])

In [ ]:
# === H? 검증 — Step ② raw plot (사람/매크로 1 trial 씩) ===
# TODO: trial_id 선택 (분포 양 끝 또는 중앙값 근처)
human_id = list_trials_by_group("lv2_human")[0]["trial_id"]
macro_id = list_trials_by_group("lv2_macro")[0]["trial_id"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for row, (label, tid) in enumerate([("human", human_id), ("macro", macro_id)]):
    trial = load_trial(tid)
    plot_trajectory(trial,      ax=axes[row, 0])
    plot_speed_over_time(trial, ax=axes[row, 1])
    plot_dt_distribution(trial, ax=axes[row, 2])
    plot_pre_click_paths(trial, ax=axes[row, 3])
    axes[row, 0].set_ylabel(f"{label}\n{axes[row, 0].get_ylabel()}", fontsize=9)
fig.suptitle(f"H? raw eventRows  (human={human_id}, macro={macro_id})", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

### H? 관찰 결과 (Step ③)

**Metric 값**: TODO  
**Raw 관찰**: TODO  
**일치 여부**: ☐ 일치 / ☐ 부분 일치 / ☐ 불일치 / ☐ 트리비얼 의심  
**메모 (1 줄)**: TODO

## 검증된 가설 (아래에 누적)

(빈 상태 — 가설 검증 시 위 템플릿을 복사하여 이 헤더 아래에 추가, 가설 테이블의 "검증 상태" 컬럼도 갱신)